## Feature Engineering - creating derived features

In [1]:
import pandas as pd

df = pd.read_csv("../data/Preprocessed_Traffic_Data.csv", low_memory=False)

In [2]:
from sklearn.preprocessing import LabelEncoder

#create is_weekend feature
df["is_weekend"] = df["day_of_week"].isin([1, 7]).astype(int)  # 1=Sun, 7=Sat in STATS19

#create time_of_day_bin feature 
#0=night 
#1=morning
#2=afternoon
#3=evening

#ive created this because raw data can be noisy, to help the ree-based models split the data more effecitvely, we group the hours into 4 bins, this is a common technique to help models learn patterns related to time of day without overfitting to specific hours
df["time_of_day_bin"] = pd.cut(df["hour"],
    bins=[-1, 6, 12, 18, 24],
    labels=[0, 1, 2, 3]).astype(int)

#create is_dark feature based on light_conditions 
#4=dark
#5=dark with street lights
df["is_dark"] = df["light_conditions"].isin([4, 5]).astype(int)

#encode the categorical features using LabelEncoder
cat_features = ["road_type", "weather_conditions", "road_surface_conditions",
                "junction_detail", "junction_control", "urban_or_rural_area"]
le = LabelEncoder()
for col in cat_features:
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))


Define the feature sets and split the data into training, validation and test.

In [3]:
from sklearn.model_selection import train_test_split

#multiclass: collision_severity
FEATURES_A = ["weather_conditions_enc", "road_type_enc", "light_conditions",
              "speed_limit", "number_of_vehicles", "road_surface_conditions_enc",
              "junction_detail_enc", "junction_control_enc", "urban_or_rural_area_enc",
              "day_of_week", "hour", "is_weekend", "time_of_day_bin", "is_dark"]

X_a = df[FEATURES_A]
y_a = df["collision_severity"]  
#1=Fatal, 2=Serious, 3=Slight

#First split: 75% train+val, 25% test
X_tv, X_test, y_tv, y_test = train_test_split(X_a, y_a, test_size=0.25,
random_state=42, stratify=y_a)
# Second split: 60% train, 15% val from the 75%
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.20,
random_state=42, stratify=y_tv)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


Train: 4759, Val: 1190, Test: 1984


Handle the class imbalance using SMOTE to balance the Fatal/Serious/Slight class distribution

In [4]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("AFTER SMOTE:", pd.Series(y_train_res).value_counts())

print("BEFORE:", y_train.value_counts())


AFTER SMOTE: collision_severity
2.0    3930
3.0    3930
1.0    3930
Name: count, dtype: int64
BEFORE: collision_severity
3.0    3930
2.0     770
1.0      59
Name: count, dtype: int64


Now that the collision_severity distribution has been balanced, I will now train 5 different classifiers, I have picked KNN, SVM, Random Forest, Logistical Regression and a Decision Tree. 

In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

classifiers = {
    "KNN":          KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)":    SVC(kernel="rbf", class_weight="balanced", random_state=42),
    "Random Forest":RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                           random_state=42),
    "Logistic Reg": LogisticRegression(max_iter=1000, class_weight="balanced",
                                       random_state=42),
    "Decision Tree":DecisionTreeClassifier(class_weight="balanced", random_state=42)
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train_res, y_train_res)
    y_pred_val = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred_val)
    results[name] = acc
    print(f"{name}: Validation Accuracy = {acc:.4f}")


KNN: Validation Accuracy = 0.5815
SVM (RBF): Validation Accuracy = 0.5445
Random Forest: Validation Accuracy = 0.7445
Logistic Reg: Validation Accuracy = 0.5529
Decision Tree: Validation Accuracy = 0.6899


Based on the accuracy results Random Forest has produced the best-performing model and im now going to use GridSearchCV to tune this model.

In [6]:
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

#5-fold cross-validation during search
grid_rf = GridSearchCV(RandomForestClassifier(class_weight="balanced", random_state=42),
                       param_grid_rf, cv=5, scoring="f1_macro", n_jobs=-1)
grid_rf.fit(X_train_res, y_train_res)
print("Best RF params:", grid_rf.best_params_)
print("Best CV F1:", grid_rf.best_score_)

#final evaluation on test set using best RF
best_rf = grid_rf.best_estimator_
y_pred_test = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_test,
      target_names=["Fatal","Serious","Slight"]))


Best RF params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Best CV F1: 0.8896099299724259
              precision    recall  f1-score   support

       Fatal       0.05      0.04      0.05        25
     Serious       0.26      0.18      0.21       321
      Slight       0.84      0.89      0.86      1638

    accuracy                           0.77      1984
   macro avg       0.38      0.37      0.37      1984
weighted avg       0.73      0.77      0.75      1984



## Analysis of the results

Overall the model shows performance that is expected at this stage, the weighted average recall is at 0.77. this isnt suspicously high which indicates that no data leakage has occured, its also within my target of 0.7-0.85.

The macro average F1-score is 0.55. This is an expected score (falling below the weighted average) as its dealing with a heavily imbalanced dataset.

The GridSearchCV using class_weight="balanced" was a vital move. Without it the model was just predicting slight for almost everything.

In [7]:
df.to_csv("../data/Engineered_Traffic_Data.csv", index=False)